In [16]:
!pip install sentence-transformers scikit-learn python-dotenv

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [17]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

WIKI = Path(os.environ["WIKI_PATH"])

notes = []

for file in WIKI.rglob("*.md"):
    text = file.read_text(encoding="utf-8")
    notes.append({"path": str(file), "text": text})

print(f"Loaded {len(notes)} notes")
print(notes[0]["path"])
print(notes[0]["text"][:300])

Loaded 90 notes
/Users/sabimantock/Library/Mobile Documents/iCloud~md~obsidian/Documents/My Brain/wiki/index.md
# Wiki Index

Content catalog — updated after every ingest.

---

## Sources

- [[wiki/sources/ai-2027]] — Research-backed scenario forecasting AGI by 2027; covers timelines, alignment failure, US-China race, and intelligence explosion (1 source)
- [[wiki/sources/ai-in-engineering-education-review]]


In [18]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")   # the thing that draws the meaning map

texts = [n["text"] for n in notes]                # just the text of each note
embeddings = model.encode(texts, show_progress_bar=True)

print(embeddings.shape)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

(90, 384)


In [19]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

question = "what is the difference between stemming and lemmatization?"
q_emb = model.encode([question])                    # the question becomes a dot too

scores = cosine_similarity(q_emb, embeddings)[0]     # how close it is to every note
top = np.argsort(scores)[::-1][:3]                   # the 3 nearest dots

for i in top:
    print(round(float(scores[i]), 3), notes[i]["path"].split("/")[-1])

0.735 nlp-lemmatization.md
0.625 nlp-lemmatization.md
0.583 nlp-stemming.md


In [20]:
!pip install anthropic

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [22]:
import anthropic

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# 1. glue the top retrieved notes together into one block of context
context = "\n\n---\n\n".join(notes[i]["text"] for i in top)

# 2. build the prompt: give it the notes, then the question
prompt = f"""Answer the question using ONLY the notes below.
If the answer isn't in the notes, say you don't know.

NOTES:
{context}

QUESTION: {question}"""

# 3. send it to the LLM and print the grounded answer
resp = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=500,
    messages=[{"role": "user", "content": prompt}],
)
print(resp.content[0].text)

## Stemming vs. Lemmatization

The core difference is **how** each finds the root form of a word:

- **Stemming** chops off endings using rules — no dictionary involved. It's fast but crude, and the result may not be a real word (e.g., "studies" → "studi").
- **Lemmatization** looks the word up in a dictionary (WordNet in NLTK) and returns the correct base form — always a real word that preserves meaning (e.g., "studies" → "study").

---

Here's the full comparison:

| | Stemming | Lemmatization |
|---|---|---|
| Uses dictionary | No — rule-based | Yes — WordNet |
| Output | May not be a real word | Always a real word |
| Preserves meaning | No | Yes |
| Speed | Faster | Slower |
| NLTK tool | `PorterStemmer` | `WordNetLemmatizer` |
| Example | "studies" → "studi" | "studies" → "study" |

---

## When to use each

- Use **stemming** when speed matters and exact word forms don't (e.g., speed-sensitive pipelines, large datasets).
- Use **lemmatization** when word meaning matters downstre